# 5. Feature Engineering

The same deterministic feature-building logic is applied independently to the training, validation, and test sets to ensure consistency with the future production inference pipeline.

The engineered features include purchase-time information, approval duration, estimated delivery window, holiday indicators, and seller-state grouping.

No target information is used during feature construction.

### Import Libraries & Data

In [1]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"

ARTIFACTS_DIR.mkdir(exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
Path("artifacts").mkdir(exist_ok=True)
Path("data/features").mkdir(parents=True, exist_ok=True)

In [ ]:
train = pd.read_csv(PROJECT_ROOT / "data" / "train.csv")
validation = pd.read_csv(PROJECT_ROOT / "data" / "validation.csv")
test = pd.read_csv(PROJECT_ROOT / "data" / "test.csv")

In [3]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67529, 22)
Validation: (14470, 22)
Test: (14471, 22)


In [4]:
print("Train columns:")
print(train.columns.tolist())

print("\nSame columns in all splits:")
print(
    list(train.columns) == list(validation.columns) == list(test.columns)
)

Train columns:
['order_purchase_timestamp', 'order_approved_at', 'order_estimated_delivery_date', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'total_price', 'total_freight', 'item_count', 'total_payment', 'payment_installments', 'payment_count', 'seller_count', 'seller_state_count', 'seller_states', 'seller_zip_code_prefix', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'distance_km', 'is_late']

Same columns in all splits:
True


## Split Target from Features

In [5]:
X_train = train.drop(columns=["is_late"]).copy()
y_train = train["is_late"].copy()

X_val = validation.drop(columns=["is_late"]).copy()
y_val = validation["is_late"].copy()

X_test = test.drop(columns=["is_late"]).copy()
y_test = test["is_late"].copy()

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:", X_val.shape, "| y_val:", y_val.shape)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)

X_train: (67529, 21) | y_train: (67529,)
X_val: (14470, 21) | y_val: (14470,)
X_test: (14471, 21) | y_test: (14471,)


## 1. Feature Engineering

The same deterministic feature-building logic is applied to the training, validation, and test sets to ensure consistency with production.

In [6]:
import holidays

In [7]:
def build_features(df):
    df = df.copy()

    # Convert date columns
    date_cols = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_estimated_delivery_date"
    ]

    for col in date_cols:
        df[col] = pd.to_datetime(df[col])

    # Purchase time features
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_weekday"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

    # Time between purchase and approval
    df["approval_time_hours"] = (
        df["order_approved_at"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600

    # Estimated delivery window
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 3600)

    # Brazilian public holiday indicator
    years = df["order_purchase_timestamp"].dt.year.unique()
    holiday_calendar = holidays.Brazil(years=years)

    df["is_holiday"] = (
        df["order_purchase_timestamp"]
        .dt.date
        .isin(holiday_calendar)
        .astype(int)
    )

    # Group orders involving sellers from multiple states
    df["seller_states"] = np.where(
        df["seller_state_count"] == 1,
        df["seller_states"],
        "MULTI"
    )

    return df

In [8]:
X_train = build_features(X_train)
X_val = build_features(X_val)
X_test = build_features(X_test)

In [9]:
new_time_features = [
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "approval_time_hours",
    "estimated_delivery_days"
]

X_train[new_time_features].describe()

,purchase_month,purchase_weekday,purchase_hour,approval_time_hours,estimated_delivery_days
count,67529.000000,67529.000000,67529.000000,67515.000000,67529.000000
mean,5.971331,2.789809,14.779635,9.930969,24.595253
std,3.756439,1.967251,5.353652,20.421087,8.018400
min,1.000000,0.000000,0.000000,0.000000,7.005127
25%,3.000000,1.000000,11.000000,0.205278,19.543333
50%,5.000000,3.000000,15.000000,0.307500,23.621331
75%,10.000000,4.000000,19.000000,13.447639,28.564965
max,12.000000,6.000000,23.000000,741.443611,155.135463


In [10]:
print("Train:")
print(X_train["is_holiday"].value_counts())

print("\nValidation:")
print(X_val["is_holiday"].value_counts())

print("\nTest:")
print(X_test["is_holiday"].value_counts())

Train:
is_holiday
0    66458
1     1071
Name: count, dtype: int64

Validation:
is_holiday
0    14064
1      406
Name: count, dtype: int64

Test:
is_holiday
0    14471
Name: count, dtype: int64


## 2. Feature Selection

In [11]:
selected_features = [
    # Categorical
    "customer_state",
    "seller_states",

    # Order and payment
    "total_price",
    "total_freight",
    "item_count",
    "total_payment",
    "payment_installments",
    "payment_count",

    # Seller and distance
    "seller_count",
    "seller_state_count",
    "distance_km",

    # Date and time features
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "approval_time_hours",
    "estimated_delivery_days",
    "is_holiday"
]

X_train = X_train[selected_features].copy()
X_val = X_val[selected_features].copy()
X_test = X_test[selected_features].copy()

The selected features are based on the exploratory data analysis and the defined prediction-time scenario.

Only information available at or shortly after order approval is included. Information generated during shipping or after delivery is excluded to prevent data leakage.

The selected variables represent order characteristics, payment information, seller information, geographic information, and purchase-time features.

High-cardinality raw geographic variables, such as customer city and ZIP prefixes, are not directly included in the final feature set.

## 3. Missing Values Handling

Missing values are handled using transformers fitted on the training data only.  
The fitted transformers are then applied to the validation and test sets to avoid data leakage.

*Check Missing Values*

In [12]:
print("Training:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

print("\nValidation:")
print(X_val.isnull().sum()[X_val.isnull().sum() > 0])

print("\nTest:")
print(X_test.isnull().sum()[X_test.isnull().sum() > 0])

Training:
total_payment              1
payment_installments       1
payment_count              1
distance_km             1116
approval_time_hours       14
dtype: int64

Validation:
distance_km    329
dtype: int64

Test:
distance_km    300
dtype: int64


*Numerical Imputation*

In [13]:
numeric_features = [
    "total_price",
    "total_freight",
    "item_count",
    "total_payment",
    "payment_installments",
    "payment_count",
    "seller_count",
    "seller_state_count",
    "distance_km",
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "approval_time_hours",
    "estimated_delivery_days",
    "is_holiday"
]


numeric_imputer = SimpleImputer(strategy="median")

# Fit only on training data
numeric_imputer.fit(X_train[numeric_features])

# Apply the same fitted imputer to all splits
X_train[numeric_features] = numeric_imputer.transform(X_train[numeric_features])

X_val[numeric_features] = numeric_imputer.transform(X_val[numeric_features])

X_test[numeric_features] = numeric_imputer.transform(X_test[numeric_features])

In [14]:
print("Missing after numerical imputation:")
print("Train:", X_train[numeric_features].isnull().sum().sum())
print("Validation:", X_val[numeric_features].isnull().sum().sum())
print("Test:", X_test[numeric_features].isnull().sum().sum())

Missing after numerical imputation:
Train: 0
Validation: 0
Test: 0


## 4. Encoding the Categorical Features
Categorical features are encoded using One-Hot Encoding.


In [15]:
categorical_features = ["customer_state","seller_states"]

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Fit only on training data
encoder.fit(X_train[categorical_features])

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_cate

In [16]:
# Transform categorical features using the fitted encoder
train_encoded = encoder.transform(X_train[categorical_features])
val_encoded = encoder.transform(X_val[categorical_features])
test_encoded = encoder.transform(X_test[categorical_features])

# Get encoded feature names
encoded_feature_names = encoder.get_feature_names_out(categorical_features)

print("Number of encoded features:", len(encoded_feature_names))
print(encoded_feature_names)

Number of encoded features: 50
['customer_state_AC' 'customer_state_AL' 'customer_state_AM'
 'customer_state_AP' 'customer_state_BA' 'customer_state_CE'
 'customer_state_DF' 'customer_state_ES' 'customer_state_GO'
 'customer_state_MA' 'customer_state_MG' 'customer_state_MS'
 'customer_state_MT' 'customer_state_PA' 'customer_state_PB'
 'customer_state_PE' 'customer_state_PI' 'customer_state_PR'
 'customer_state_RJ' 'customer_state_RN' 'customer_state_RO'
 'customer_state_RR' 'customer_state_RS' 'customer_state_SC'
 'customer_state_SE' 'customer_state_SP' 'customer_state_TO'
 'seller_states_AM' 'seller_states_BA' 'seller_states_CE'
 'seller_states_DF' 'seller_states_ES' 'seller_states_GO'
 'seller_states_MA' 'seller_states_MG' 'seller_states_MS'
 'seller_states_MT' 'seller_states_MULTI' 'seller_states_PA'
 'seller_states_PB' 'seller_states_PE' 'seller_states_PI'
 'seller_states_PR' 'seller_states_RJ' 'seller_states_RN'
 'seller_states_RO' 'seller_states_RS' 'seller_states_SC'
 'seller_st

In [17]:
# Convert encoded arrays to DataFrames
train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)

val_encoded_df = pd.DataFrame(
    val_encoded,
    columns=encoded_feature_names,
    index=X_val.index
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

In [18]:
X_train_final = pd.concat([X_train[numeric_features], train_encoded_df], axis=1)

X_val_final = pd.concat([X_val[numeric_features], val_encoded_df], axis=1)

X_test_final = pd.concat([X_test[numeric_features], test_encoded_df], axis=1)

In [19]:
print("Final shapes:")
print("Train:", X_train_final.shape)
print("Validation:", X_val_final.shape)
print("Test:", X_test_final.shape)

print("\nMissing values:")
print("Train:", X_train_final.isnull().sum().sum())
print("Validation:", X_val_final.isnull().sum().sum())
print("Test:", X_test_final.isnull().sum().sum())

print("\nSame features in all splits:")
print(
    list(X_train_final.columns)
    == list(X_val_final.columns)
    == list(X_test_final.columns)
)

Final shapes:
Train: (67529, 65)
Validation: (14470, 65)
Test: (14471, 65)

Missing values:
Train: 0
Validation: 0
Test: 0

Same features in all splits:
True


## 5. Feature Scaling

Numerical features are standardized using `StandardScaler`.

The scaler is fitted on the training data only and then applied to the validation and test sets. One-hot encoded categorical features are not scaled.

In [20]:
features_to_scale = [
    "total_price",
    "total_freight",
    "item_count",
    "total_payment",
    "payment_installments",
    "payment_count",
    "distance_km",
    "approval_time_hours",
    "estimated_delivery_days"
]

scaler = StandardScaler()

# Fit on training data only
scaler.fit(X_train_final[features_to_scale])

# Apply the same fitted scaler to all splits
X_train_final[features_to_scale] = scaler.transform(X_train_final[features_to_scale])

X_val_final[features_to_scale] = scaler.transform(X_val_final[features_to_scale])

X_test_final[features_to_scale] = scaler.transform(X_test_final[features_to_scale])

In [21]:
X_train_final[features_to_scale].describe().round(2)

,total_price,total_freight,item_count,total_payment,payment_installments,payment_count,distance_km,approval_time_hours,estimated_delivery_days
count,67529.00,67529.00,67529.00,67529.00,67529.00,67529.00,67529.00,67529.00,67529.00
mean,-0.00,-0.00,0.00,0.00,0.00,-0.00,0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-0.65,-1.11,-0.26,-0.69,-0.72,-0.12,-1.04,-0.49,-2.19
25%,-0.44,-0.41,-0.26,-0.45,-0.72,-0.12,-0.65,-0.48,-0.63
50%,-0.25,-0.27,-0.26,-0.25,-0.35,-0.12,-0.28,-0.47,-0.12
75%,0.07,0.07,-0.26,0.08,0.37,-0.12,0.33,0.17,0.50
max,64.84,49.00,36.66,63.03,7.63,62.82,7.99,35.82,16.28


## 6. Save Feature Engineering Artifacts

The final processed feature tables and all fitted preprocessing objects are saved for use in model training and production.

The same fitted imputer, encoder, and scaler must be reused on new data without fitting them again.

In [ ]:
joblib.dump(numeric_imputer, ARTIFACTS_DIR / "numeric_imputer.joblib")

joblib.dump(encoder, ARTIFACTS_DIR / "onehot_encoder.joblib")

joblib.dump(scaler, ARTIFACTS_DIR / "scaler.joblib")

['artifacts/scaler.joblib']

In [ ]:
final_feature_list = X_train_final.columns.tolist()

joblib.dump(final_feature_list,ARTIFACTS_DIR / "feature_list.joblib")

print("Number of final model features:", len(final_feature_list))

Number of final model features: 65


In [ ]:
X_train_final.to_csv(FEATURES_DIR / "X_train.csv", index=False)

X_val_final.to_csv(FEATURES_DIR / "X_validation.csv", index=False)

X_test_final.to_csv(FEATURES_DIR / "X_test.csv", index=False)

y_train.to_csv(FEATURES_DIR / "y_train.csv", index=False)

y_val.to_csv(FEATURES_DIR / "y_validation.csv", index=False)

y_test.to_csv(FEATURES_DIR / "y_test.csv",index=False)

## 7. Final Validation

In [26]:
print("Final feature tables:")
print("Train:", X_train_final.shape)
print("Validation:", X_val_final.shape)
print("Test:", X_test_final.shape)

print("\nTargets:")
print("Train:", y_train.shape)
print("Validation:", y_val.shape)
print("Test:", y_test.shape)

print("\nMissing values:")
print("Train:", X_train_final.isnull().sum().sum())
print("Validation:", X_val_final.isnull().sum().sum())
print("Test:", X_test_final.isnull().sum().sum())

print("\nSame feature columns:")
print(
    list(X_train_final.columns)
    == list(X_val_final.columns)
    == list(X_test_final.columns)
)

print("\nNumber of final features:", len(final_feature_list))

Final feature tables:
Train: (67529, 65)
Validation: (14470, 65)
Test: (14471, 65)

Targets:
Train: (67529,)
Validation: (14470,)
Test: (14471,)

Missing values:
Train: 0
Validation: 0
Test: 0

Same feature columns:
True

Number of final features: 65
